# Text-to-Text Generator -- Schema-Driven, Batch-Based, Judge-Validated

Generates text-to-text (T2T) questions -- typed-answer format, no options.
Same quota loop as `mcq_generator.ipynb` with T2T-specific judges and models.

## Cell 1 -- Setup & Imports

In [ ]:
import sys, os, json, warnings
from pathlib import Path
from types import SimpleNamespace
from typing import Literal

warnings.filterwarnings('ignore')

# Locate project root
_candidate = Path().resolve()
if (_candidate / 'utils.py').exists():
    PROJECT_ROOT = _candidate
elif (_candidate.parent / 'utils.py').exists():
    PROJECT_ROOT = _candidate.parent
else:
    raise RuntimeError(
        f'Cannot locate project root from {_candidate}. '
        'Open Jupyter from d:/Topin or d:/Topin/notebooks.'
    )

DATA_DIR      = PROJECT_ROOT / 'data'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
ARTIFACTS_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Auto-inject venv site-packages
_venv_sp = PROJECT_ROOT / '.venv' / 'Lib' / 'site-packages'
if not _venv_sp.exists():
    _venv_sp = PROJECT_ROOT / '.venv' / 'lib' / 'site-packages'
if _venv_sp.exists() and str(_venv_sp) not in sys.path:
    sys.path.insert(0, str(_venv_sp))
    print(f'Injected venv site-packages: {_venv_sp}')

import dspy
from pydantic import BaseModel, Field

print(f'Project root : {PROJECT_ROOT}')
print(f'Data dir     : {DATA_DIR}')
print(f'Artifacts    : {ARTIFACTS_DIR}')
print(f'DSPy version : {dspy.__version__}')


## Cell 2 -- Configure DSPy (Mistral)

In [ ]:
from utils import configure_dspy_from_env

lm = configure_dspy_from_env()
print(f'Active LM : {lm}')


## Cell 3 -- Load T2T Datasets

Reads `training_dataset_clean_full.json` and `eval_dataset_clean.json`.
Builds the reference example pool used in Cell 11.

In [ ]:
DIFF_MAP = {'A1':'Easy','A2':'Easy','B1':'Medium','B2':'Medium','C1':'Hard','C2':'Hard'}
CEFR_TO_QTYPE = {
    'A1': 'comprehension',
    'A2': 'reorder',
    'B1': 'paragraph',
    'B2': 'paragraph',
    'C1': 'situational',
    'C2': 'situational',
}

_TRAIN_PATH = DATA_DIR / 'training_dataset_clean_full.json'
_EVAL_PATH  = DATA_DIR / 'eval_dataset_clean.json'

def _load_t2t_file(path: Path) -> list:
    if not path.exists():
        print(f'  [WARN] Not found: {path}')
        return []
    with open(path, encoding='utf-8') as f:
        return json.load(f)

_train_rows = _load_t2t_file(_TRAIN_PATH)
_eval_rows  = _load_t2t_file(_EVAL_PATH)

from collections import Counter
_train_counts = Counter(r['target_cefr'] for r in _train_rows)
_eval_counts  = Counter(r['target_cefr'] for r in _eval_rows)

print('Training dataset:')
for cefr in ['A1','A2','B1','B2','C1','C2']:
    print(f'  {cefr} ({DIFF_MAP[cefr]:<6}): {_train_counts.get(cefr, 0)} questions')
print(f'  Total: {len(_train_rows)}')

print('\nEval dataset:')
for cefr in ['A1','A2','B1','B2','C1','C2']:
    print(f'  {cefr} ({DIFF_MAP[cefr]:<6}): {_eval_counts.get(cefr, 0)} questions')
print(f'  Total: {len(_eval_rows)}')


## Cell 4 -- Input Models

- `SubtopicRequirement` -- per-CEFR counts (A1 to C2)
- `InputSchema` -- topic + subtopics + constraints
- `ExampleT2TQuestion` / `ExampleT2TQuestionSet` -- reference examples

In [ ]:
class SubtopicRequirement(BaseModel):
    '''Per-subtopic question counts for each of the 6 CEFR levels.

    CEFR mapping:
        A1, A2 -> Easy   (comprehension / reorder)
        B1, B2 -> Medium (paragraph from notes)
        C1, C2 -> Hard   (situational / extended writing)
    '''
    subtopic:  str
    a1_count:  int = 0
    a2_count:  int = 0
    b1_count:  int = 0
    b2_count:  int = 0
    c1_count:  int = 0
    c2_count:  int = 0

    @property
    def easy_count(self) -> int:   return self.a1_count + self.a2_count
    @property
    def medium_count(self) -> int: return self.b1_count + self.b2_count
    @property
    def hard_count(self) -> int:   return self.c1_count + self.c2_count
    @property
    def total(self) -> int:        return self.easy_count + self.medium_count + self.hard_count


class GenerationConstraints(BaseModel):
    questions_per_iteration:       int = 5
    max_iterations_per_difficulty: int = 20


class InputSchema(BaseModel):
    topic:       str
    subtopics:   list[SubtopicRequirement]
    constraints: GenerationConstraints = Field(default_factory=GenerationConstraints)


class ExampleT2TQuestion(BaseModel):
    stem:            str
    expected_answer: str
    explanation:     str
    difficulty:      str | None = None
    cefr:            str | None = None
    subtopic:        str | None = None
    question_type:   str | None = None


class ExampleT2TQuestionSet(BaseModel):
    items: list[ExampleT2TQuestion] = Field(default_factory=list)

    def filter_examples(
        self,
        *,
        subtopic:   str,
        difficulty: str,
        cefr:       str | None = None,
    ) -> list[ExampleT2TQuestion]:
        '''Priority 1: match subtopic + exact CEFR.
           Priority 2: match subtopic + difficulty band.
           Priority 3: match difficulty band only.
           Priority 4: return all (fallback).
        '''
        if cefr:
            p1 = [q for q in self.items
                  if q.cefr == cefr and q.subtopic in (None, subtopic)]
            if p1:
                return p1
        p2 = [q for q in self.items
              if q.difficulty in (None, difficulty)
              and q.subtopic in (None, subtopic)]
        if p2:
            return p2
        p3 = [q for q in self.items if q.difficulty in (None, difficulty)]
        if p3:
            return p3
        return self.items


print('Input models defined.')
print('  SubtopicRequirement: a1_count, a2_count, b1_count, b2_count, c1_count, c2_count')
print('  CEFR: A1/A2=Easy | B1/B2=Medium | C1/C2=Hard')


## Cell 5 -- Output Models

- `T2TItem` -- one accepted T2T question (no options, has `expected_answer`)
- `T2TQuestionStore` -- grouped by difficulty with `count_by_cefr()`
- `T2TGenerationResult` -- store + rejected + warnings

In [ ]:
class T2TItem(BaseModel):
    question_number:   int
    topic:             str
    subtopic:          str | None
    target_cefr:       str
    target_difficulty: str
    question_type:     str
    stem:              str
    expected_answer:   str
    explanation:       str


class T2TQuestionStore(BaseModel):
    '''Accepted T2T questions grouped by difficulty band.'''
    easy:   list[T2TItem] = Field(default_factory=list)
    medium: list[T2TItem] = Field(default_factory=list)
    hard:   list[T2TItem] = Field(default_factory=list)

    def add(self, item: T2TItem) -> None:
        d = item.target_difficulty.strip().lower()
        if d == 'easy':   self.easy.append(item)
        elif d == 'medium': self.medium.append(item)
        elif d == 'hard':   self.hard.append(item)
        else: raise ValueError(f'Unknown difficulty: {item.target_difficulty}')

    def get_used_stems(self) -> list[str]:
        return [q.stem for q in self.easy + self.medium + self.hard]

    def count(self, difficulty: str) -> int:
        d = difficulty.strip().lower()
        if d == 'easy':   return len(self.easy)
        if d == 'medium': return len(self.medium)
        if d == 'hard':   return len(self.hard)
        raise ValueError(f'Unknown difficulty: {difficulty}')

    def count_by_cefr(self, cefr: str) -> int:
        return sum(1 for q in self.easy + self.medium + self.hard
                   if q.target_cefr == cefr)

    def all_items(self) -> list[T2TItem]:
        return self.easy + self.medium + self.hard


class T2TGenerationResult(BaseModel):
    store:    T2TQuestionStore
    rejected: list[dict] = Field(default_factory=list)
    warnings: list[str]  = Field(default_factory=list)


print('Output models defined.')
print('  T2TItem: no options -- has expected_answer instead')
print('  T2TQuestionStore: easy/medium/hard + count_by_cefr()')


## Cell 6 -- T2T Generator Signature + Agent

**Typed batch:** `T2TGenerationRequest` -> `T2TGenerationBatch`

`T2TGeneratorAgent.forward()` owns the full quota loop:
Generate -> Hard-validate -> DifficultyJudge (batch) -> RubricJudge (batch) -> Save

In [ ]:
class T2TGenerationRequest(BaseModel):
    topic:              str
    subtopic:           str
    target_cefr:        str
    target_difficulty:  str
    question_type:      str
    example_questions:  list[ExampleT2TQuestion]
    already_used_stems: list[str]
    batch_size:         int


class T2TGeneratedQuestion(BaseModel):
    stem:            str
    expected_answer: str
    explanation:     str


class T2TGenerationBatch(BaseModel):
    questions: list[T2TGeneratedQuestion]


class T2TGeneratorSignature(dspy.Signature):
    '''Generate a batch of text-to-text (T2T) questions for language learners.

    T2T questions require a typed free-text response -- NO options list.
    Match the question_type exactly:
      comprehension : One simple sentence + factual question (A1)
      reorder       : Jumbled words to arrange OR one-sentence situation reply (A2)
      paragraph     : Bullet notes + write 35-60 word paragraph (B1/B2)
      situational   : Complex situation requiring formal/extended response (C1/C2)

    Use example_questions as a style and format reference for the CEFR level.

    STRICT RULES:
    - stem must clearly state the task and context
    - expected_answer must CORRECTLY answer the stem
    - explanation must justify why expected_answer is correct
    - each stem must differ from all stems in already_used_stems
    - vocabulary and task complexity must match target_cefr
    - Return exactly batch_size questions
    '''
    request: T2TGenerationRequest = dspy.InputField(
        desc='Batch parameters: topic, CEFR, question_type, reference examples, used stems'
    )
    output: T2TGenerationBatch = dspy.OutputField(
        desc='Batch of T2T questions -- stem + expected_answer + explanation'
    )


class T2TGeneratorAgent(dspy.Module):
    '''Generates T2T questions and runs the quota loop inside forward().

    Dependencies injected at construction:
      self.store            -- T2TQuestionStore
      self.difficulty_judge -- T2TDifficultyJudgeWrapper (batch)
      self.rubric_judge     -- T2TRubricJudgeWrapper (batch)
    '''

    def __init__(self, *, store, difficulty_judge, rubric_judge):
        super().__init__()
        self.generate         = dspy.ChainOfThought(T2TGeneratorSignature)
        self.store            = store
        self.difficulty_judge = difficulty_judge
        self.rubric_judge     = rubric_judge

    def forward(
        self,
        *,
        schema:            InputSchema,
        example_questions: ExampleT2TQuestionSet,
        topic:             str,
        subtopic:          str | None,
        target_cefr:       str,
        target_difficulty: str,
        target_count:      int,
        rejected:          list,
        warnings:          list,
    ) -> None:
        '''Quota loop: iterate until store.count_by_cefr(target_cefr) >= target_count.

        Each iteration:
          1. self.generate(request) -- ChainOfThought LLM call -> T2TGenerationBatch
          2. hard_validate_t2t() all questions in batch
          3. self.difficulty_judge(valid_items) -- batch LLM call
          4. self.rubric_judge(diff_passed)     -- batch LLM call
          5. self.store.add(accepted)
        '''
        if target_count <= 0:
            return

        q_type    = CEFR_TO_QTYPE.get(target_cefr, 'comprehension')
        iteration = 0
        q_number  = len(self.store.all_items()) + 1

        print(f'\n[{target_difficulty} / {target_cefr} / {q_type}]  '
              f'target={target_count}  subtopic={subtopic!r}')

        while self.store.count_by_cefr(target_cefr) < target_count:
            iteration += 1
            if iteration > schema.constraints.max_iterations_per_difficulty:
                msg = (f'{target_difficulty}/{target_cefr}: max iterations '
                       f'({schema.constraints.max_iterations_per_difficulty}) reached. '
                       f'Accepted {self.store.count_by_cefr(target_cefr)}/{target_count}.')
                warnings.append(msg)
                print(f'  [WARNING] {msg}')
                break

            needed   = target_count - self.store.count_by_cefr(target_cefr)
            batch_sz = min(schema.constraints.questions_per_iteration, needed)
            relevant = example_questions.filter_examples(
                subtopic=subtopic or '',
                difficulty=target_difficulty,
                cefr=target_cefr,
            )
            request = T2TGenerationRequest(
                topic=topic,
                subtopic=subtopic or '',
                target_cefr=target_cefr,
                target_difficulty=target_difficulty,
                question_type=q_type,
                example_questions=relevant,
                already_used_stems=self.store.get_used_stems(),
                batch_size=batch_sz,
            )

            # 1. Generate batch
            print(f'  [Iter {iteration}] Generating {batch_sz} {q_type} questions...')
            try:
                pred  = self.generate(request=request)
                batch = pred.output.questions
            except Exception as e:
                rejected.append({
                    'stage': 'generation', 'cefr': target_cefr,
                    'difficulty': target_difficulty,
                    'iteration': iteration, 'error': str(e),
                })
                print(f'  [Gen Error] iter={iteration}: {e}')
                continue

            # 2. Hard-validate entire batch
            valid_items = []
            for q in batch:
                errors = hard_validate_t2t(q)
                if errors:
                    rejected.append({
                        'stage': 'hard_validate', 'cefr': target_cefr,
                        'difficulty': target_difficulty,
                        'iteration': iteration, 'stem': q.stem[:60], 'errors': errors,
                    })
                    print(f'  [Hard Fail] {errors}')
                    continue
                valid_items.append(T2TItem(
                    question_number=q_number,
                    topic=topic,
                    subtopic=subtopic,
                    target_cefr=target_cefr,
                    target_difficulty=target_difficulty,
                    question_type=q_type,
                    stem=q.stem,
                    expected_answer=q.expected_answer,
                    explanation=q.explanation,
                ))
                q_number += 1

            if not valid_items:
                print(f'  [Skip] All {len(batch)} failed hard validation.')
                continue
            print(f'  [Hard]  {len(valid_items)}/{len(batch)} passed.')

            # 3. DifficultyJudge -- one batch call
            try:
                diff_results = self.difficulty_judge(
                    items=valid_items,
                    expected_difficulty=target_difficulty,
                )
            except Exception as e:
                rejected.append({'stage': 'difficulty_judge_error', 'error': str(e)})
                print(f'  [Diff Error] {e}')
                continue

            diff_passed = []
            for item, diff in zip(valid_items, diff_results):
                if diff.passed:
                    diff_passed.append(item)
                else:
                    rejected.append({
                        'stage': 'difficulty', 'q': item.question_number,
                        'cefr': target_cefr, 'difficulty': target_difficulty,
                        'reason': diff.reason, 'stem': item.stem[:60],
                    })
                    print(f'  [Diff Fail] Q{item.question_number}: {diff.reason}')

            print(f'  [Diff]  {len(diff_passed)}/{len(valid_items)} passed.')
            if not diff_passed:
                continue

            # 4. RubricJudge -- one batch call
            try:
                rub_results = self.rubric_judge(items=diff_passed)
            except Exception as e:
                rejected.append({'stage': 'rubric_judge_error', 'error': str(e)})
                print(f'  [Rub Error] {e}')
                continue

            accepted_count = 0
            for item, rub in zip(diff_passed, rub_results):
                if not rub.passed:
                    rejected.append({
                        'stage': 'rubric', 'q': item.question_number,
                        'cefr': target_cefr, 'difficulty': target_difficulty,
                        'reason': rub.reason, 'stem': item.stem[:60],
                    })
                    print(f'  [Rub  Fail] Q{item.question_number}: {rub.reason}')
                    continue

                # 5. Save to store
                self.store.add(item)
                accepted_count += 1
                total = self.store.count_by_cefr(target_cefr)
                print(f'  [Accepted]  Q{item.question_number} '
                      f'({target_cefr}/{target_difficulty}/{item.question_type}) '
                      f'{total}/{target_count}')
                if total >= target_count:
                    break

            print(f'  [Rubric] {accepted_count}/{len(diff_passed)} passed.')


print('T2TGeneratorAgent defined.')
print('  self.generate = ChainOfThought(T2TGeneratorSignature)')
print('  forward() -- quota loop: Generate->Validate->DiffJudge->RubricJudge->Save')


## Cell 7 -- T2T Difficulty Judge + Rubric Judge

New T2T-specific judges (no options field -- adapted for typed-answer questions).
Loads from `artifacts/t2t_difficulty_optimized.json` and `artifacts/t2t_rubric_optimized.json` if present.

In [ ]:
# Shared DifficultyResult / DifficultyOutput
class DifficultyResult(BaseModel):
    question_id:          str
    predicted_cefr:       Literal['A1', 'A2', 'B1', 'B2', 'C1', 'C2']
    predicted_difficulty: Literal['Easy', 'Medium', 'Hard']

class DifficultyOutput(BaseModel):
    results: list[DifficultyResult]


# T2T Difficulty Judge
class T2TQuestion(BaseModel):
    '''T2T input for DifficultyJudge -- no options, uses expected_answer.'''
    question_id:       str
    stem:              str
    expected_answer:   str
    explanation:       str
    target_cefr:       str
    target_difficulty: str


class T2TDifficultySignature(dspy.Signature):
    '''Classify text-to-text questions by CEFR level and difficulty.

    For each question analyse:
      - Vocabulary complexity in stem and expected_answer
      - Task type complexity (factual recall vs extended writing)
      - Length and structure of expected response
      - Grammar structures required

    A1/A2 -> Easy   (comprehension / word reorder / one-sentence reply)
    B1/B2 -> Medium (paragraph from notes, 35-60 words)
    C1/C2 -> Hard   (formal extended response, complex argumentation)

    Return one DifficultyResult per question in the same order.
    '''
    questions: list[T2TQuestion]  = dspy.InputField(desc='T2T questions to classify')
    output:    DifficultyOutput   = dspy.OutputField(desc='Classification results')


class T2TDifficultyAgent(dspy.Module):
    def __init__(self):
        super().__init__()
        self.judge = dspy.ChainOfThought(T2TDifficultySignature)
    def forward(self, questions):
        return self.judge(questions=questions)


# T2T Rubric Judge
class T2TRubricQuestion(BaseModel):
    question_id:       str
    stem:              str
    expected_answer:   str
    explanation:       str
    target_cefr:       str
    target_difficulty: str
    language_variant:  str


class T2TRubricResult(BaseModel):
    question_id:            str
    stem_clarity:           Literal['Clear', 'Needs Improvement', 'Unclear']
    grammatical_accuracy:   Literal['No Issues', 'Minor Issues', 'Major Issues']
    spelling:               Literal['No Issues', 'Minor Issues', 'Major Issues']
    answer_relevance:       Literal['Relevant', 'Partially Relevant', 'Not Relevant']
    answer_completeness:    Literal['Complete', 'Incomplete', 'Missing']
    cefr_appropriateness:   Literal['Appropriate', 'Too Easy', 'Too Hard']
    explanation_accuracy:   Literal['Accurate', 'Partially Accurate', 'Inaccurate']
    formatting_punctuation: Literal['No Issues', 'Minor Issues', 'Major Issues']
    overall_decision:       Literal['Pass', 'Revise', 'Fail']
    priority_reason:        str
    revision_feedback:      str


class T2TRubricOutput(BaseModel):
    results: list[T2TRubricResult]


class T2TRubricSignature(dspy.Signature):
    '''Evaluate text-to-text questions using the rubric.

    Priority criteria (any of these forces Fail immediately):
      - answer_relevance = Not Relevant
      - stem_clarity     = Unclear

    Evaluate:
      stem_clarity           -- Is the task instruction unambiguous?
      grammatical_accuracy   -- Grammar in stem and expected_answer
      spelling               -- Spelling in stem and expected_answer
      answer_relevance       -- Does expected_answer correctly address the stem?
      answer_completeness    -- Is the answer complete (not a fragment)?
      cefr_appropriateness   -- Is task difficulty right for target CEFR?
      explanation_accuracy   -- Does explanation justify expected_answer?
      formatting_punctuation -- Proper punctuation and capitalisation

    Return one T2TRubricResult per question in the same order.
    '''
    questions: list[T2TRubricQuestion] = dspy.InputField(desc='T2T questions to evaluate')
    output:    T2TRubricOutput         = dspy.OutputField(desc='Rubric evaluation results')


class T2TRubricAgent(dspy.Module):
    def __init__(self):
        super().__init__()
        self.judge = dspy.ChainOfThought(T2TRubricSignature)
    def forward(self, questions):
        return self.judge(questions=questions)


# Load from artifacts
T2T_DIFF_ARTIFACT = ARTIFACTS_DIR / 't2t_difficulty_optimized.json'
T2T_RUB_ARTIFACT  = ARTIFACTS_DIR / 't2t_rubric_optimized.json'

_t2t_diff_agent = T2TDifficultyAgent()
if T2T_DIFF_ARTIFACT.exists():
    _t2t_diff_agent.load(str(T2T_DIFF_ARTIFACT))
    print(f'Loaded T2T DifficultyJudge from: {T2T_DIFF_ARTIFACT}')
else:
    print(f'T2T DifficultyJudge: using unoptimised agent (artifact not found)')

_t2t_rub_agent = T2TRubricAgent()
if T2T_RUB_ARTIFACT.exists():
    _t2t_rub_agent.load(str(T2T_RUB_ARTIFACT))
    print(f'Loaded T2T RubricJudge from   : {T2T_RUB_ARTIFACT}')
else:
    print(f'T2T RubricJudge  : using unoptimised agent (artifact not found)')


# Batch judge wrappers
class T2TDifficultyJudgeWrapper:
    '''Wraps T2TDifficultyAgent. Accepts list[T2TItem], returns list[SimpleNamespace].'''
    def __call__(self, *, items, expected_difficulty):
        questions = [
            T2TQuestion(
                question_id=str(item.question_number),
                stem=item.stem,
                expected_answer=item.expected_answer,
                explanation=item.explanation,
                target_cefr=item.target_cefr,
                target_difficulty=item.target_difficulty,
            )
            for item in items
        ]
        pred = _t2t_diff_agent(questions=questions)
        return [
            SimpleNamespace(
                passed=res.predicted_difficulty.lower() == expected_difficulty.lower(),
                reason=f'predicted_cefr={res.predicted_cefr} predicted_difficulty={res.predicted_difficulty}',
            )
            for res in pred.output.results
        ]


class T2TRubricJudgeWrapper:
    '''Wraps T2TRubricAgent. Accepts list[T2TItem], returns list[SimpleNamespace].'''
    def __init__(self, language_variant='British English'):
        self.language_variant = language_variant

    def __call__(self, *, items):
        questions = [
            T2TRubricQuestion(
                question_id=str(item.question_number),
                stem=item.stem,
                expected_answer=item.expected_answer,
                explanation=item.explanation,
                target_cefr=item.target_cefr,
                target_difficulty=item.target_difficulty,
                language_variant=self.language_variant,
            )
            for item in items
        ]
        pred = _t2t_rub_agent(questions=questions)
        return [
            SimpleNamespace(
                passed=res.overall_decision == 'Pass',
                reason=res.priority_reason,
            )
            for res in pred.output.results
        ]


difficulty_judge = T2TDifficultyJudgeWrapper()
rubric_judge     = T2TRubricJudgeWrapper(language_variant='British English')

print('T2T batch judge wrappers ready.')
print('  difficulty_judge(items=[...], expected_difficulty=...) -> list[.passed/.reason]')
print('  rubric_judge(items=[...])                              -> list[.passed/.reason]')


## Cell 8 -- hard_validate_t2t Helper

Structural checks before sending to judges (no options check -- T2T has no options).

In [ ]:
def hard_validate_t2t(q: T2TGeneratedQuestion) -> list:
    '''Returns list of error strings. Empty list = structurally valid.'''
    errors = []
    if not q.stem or not q.stem.strip():
        errors.append('stem is empty')
    if not q.expected_answer or not q.expected_answer.strip():
        errors.append('expected_answer is empty')
    if len(q.expected_answer.strip()) < 2:
        errors.append('expected_answer too short (< 2 chars)')
    if not q.explanation or not q.explanation.strip():
        errors.append('explanation is empty')
    return errors


print('hard_validate_t2t() defined.')
print('  Checks: stem not empty | expected_answer >= 2 chars | explanation not empty')
print('  (No options check -- T2T format has no options)')


## Cell 9 -- Per-Difficulty Agents

Thin wrappers binding `difficulty_name`. Call `T2TGeneratorAgent.forward()`.

In [ ]:
_CEFR_TO_DIFFICULTY = {
    'A1': 'Easy',  'A2': 'Easy',
    'B1': 'Medium', 'B2': 'Medium',
    'C1': 'Hard',  'C2': 'Hard',
}


class BaseDifficultyAgent:
    '''Binds difficulty_name. Delegates to T2TGeneratorAgent.forward().'''
    difficulty_name: str = ''
    default_cefr:    str = ''

    def __init__(self, *, generator):
        self.generator = generator

    def generate_questions(
        self,
        *,
        schema, example_questions, topic, subtopic,
        target_cefr, target_count, rejected, warnings,
    ) -> None:
        self.generator(
            schema=schema,
            example_questions=example_questions,
            topic=topic,
            subtopic=subtopic,
            target_cefr=target_cefr,
            target_difficulty=self.difficulty_name,
            target_count=target_count,
            rejected=rejected,
            warnings=warnings,
        )


class EasyT2TAgent(BaseDifficultyAgent):
    difficulty_name = 'Easy'
    default_cefr    = 'A2'

class MediumT2TAgent(BaseDifficultyAgent):
    difficulty_name = 'Medium'
    default_cefr    = 'B1'

class HardT2TAgent(BaseDifficultyAgent):
    difficulty_name = 'Hard'
    default_cefr    = 'B2'


print('Per-difficulty agents defined.')
print('  EasyT2TAgent   difficulty=Easy,   cefr default=A2')
print('  MediumT2TAgent difficulty=Medium, cefr default=B1')
print('  HardT2TAgent   difficulty=Hard,   cefr default=B2')


## Cell 10 -- Orchestrator

`T2TGenerationOrchestrator` wires all components and iterates A1->A2->B1->B2->C1->C2.

In [ ]:
_CEFR_LEVELS = ['A1', 'A2', 'B1', 'B2', 'C1', 'C2']


class T2TGenerationOrchestrator:
    '''Creates and wires all components, then runs the generation loop.

    Ownership:
      self.store     -- T2TQuestionStore (shared via T2TGeneratorAgent)
      self.generator -- T2TGeneratorAgent (owns store + judges)
      self.easy/medium/hard_agent -- per-difficulty wrappers
    '''

    def __init__(self, *, difficulty_judge, rubric_judge):
        self.store = T2TQuestionStore()
        self.generator = T2TGeneratorAgent(
            store=self.store,
            difficulty_judge=difficulty_judge,
            rubric_judge=rubric_judge,
        )
        self.easy_agent   = EasyT2TAgent(generator=self.generator)
        self.medium_agent = MediumT2TAgent(generator=self.generator)
        self.hard_agent   = HardT2TAgent(generator=self.generator)
        self._agents = {
            'Easy':   self.easy_agent,
            'Medium': self.medium_agent,
            'Hard':   self.hard_agent,
        }

    def run(self, *, schema, example_questions):
        rejected = []
        warnings = []

        for req in schema.subtopics:
            SEP = '=' * 60
            print('\n' + SEP)
            print(f'Topic: {schema.topic}  |  Subtopic: {req.subtopic}')
            print(f'  Easy:   A1={req.a1_count}  A2={req.a2_count}')
            print(f'  Medium: B1={req.b1_count}  B2={req.b2_count}')
            print(f'  Hard:   C1={req.c1_count}  C2={req.c2_count}')
            print(SEP)

            for cefr in _CEFR_LEVELS:
                count_attr   = cefr.lower() + '_count'
                target_count = getattr(req, count_attr)
                if target_count == 0:
                    continue
                difficulty = _CEFR_TO_DIFFICULTY[cefr]
                agent      = self._agents[difficulty]
                agent.generate_questions(
                    schema=schema,
                    example_questions=example_questions,
                    topic=schema.topic,
                    subtopic=req.subtopic,
                    target_cefr=cefr,
                    target_count=target_count,
                    rejected=rejected,
                    warnings=warnings,
                )

        return T2TGenerationResult(
            store=self.store,
            rejected=rejected,
            warnings=warnings,
        )


print('T2TGenerationOrchestrator defined.')
print('  __init__(difficulty_judge, rubric_judge)')
print('  run(schema, example_questions) -> T2TGenerationResult')


## Cell 11 -- Define Schema + Examples, Then Run

**Edit this cell** to change topic, subtopics, and counts.
Reference examples are auto-built from the loaded datasets (Cell 3).

In [ ]:
# Build example pool from loaded datasets
def _rows_to_examples(rows):
    examples = []
    for row in rows:
        cefr  = row.get('target_cefr', '')
        diff  = DIFF_MAP.get(cefr, '')
        qtype = CEFR_TO_QTYPE.get(cefr, 'comprehension')
        td    = row.get('target_difficulty') or diff
        examples.append(ExampleT2TQuestion(
            stem=row['stem'],
            expected_answer=row['expected_answer'],
            explanation=row.get('explanation', ''),
            difficulty=diff or td,
            cefr=cefr,
            subtopic=None,
            question_type=qtype,
        ))
    return examples

_all_examples = _rows_to_examples(_train_rows) + _rows_to_examples(_eval_rows)
example_questions = ExampleT2TQuestionSet(items=_all_examples)
print(f'Reference pool: {len(_all_examples)} examples')

# Define schema -- edit counts per CEFR level as needed
schema = InputSchema(
    topic='English Language Skills',
    subtopics=[
        SubtopicRequirement(
            subtopic='Reading and Writing',
            a1_count=1,    # comprehension questions
            a2_count=1,    # word reorder / one-sentence reply
            b1_count=1,    # paragraph from notes (35-45 words)
            b2_count=0,
            c1_count=0,
            c2_count=0,
        )
    ],
    constraints=GenerationConstraints(
        questions_per_iteration=5,
        max_iterations_per_difficulty=10,
    ),
)

# Build orchestrator and run
orchestrator = T2TGenerationOrchestrator(
    difficulty_judge=difficulty_judge,
    rubric_judge=rubric_judge,
)

result = orchestrator.run(schema=schema, example_questions=example_questions)

# Summary
SEP = '=' * 60
print()
print(SEP)
print('GENERATION COMPLETE')
print(SEP)

for req in schema.subtopics:
    print(f'\nSubtopic: {req.subtopic}')
    for cefr in _CEFR_LEVELS:
        count_attr = cefr.lower() + '_count'
        target     = getattr(req, count_attr)
        if target == 0:
            continue
        accepted = result.store.count_by_cefr(cefr)
        status   = 'OK' if accepted >= target else 'PARTIAL'
        diff     = _CEFR_TO_DIFFICULTY[cefr]
        qtype    = CEFR_TO_QTYPE[cefr]
        print(f'  {cefr} ({diff:<6} / {qtype:<15}): {accepted}/{target}  [{status}]')

print(f'\nTotal accepted : {len(result.store.all_items())}')
print(f'Total rejected : {len(result.rejected)}')
if result.warnings:
    print('Warnings:')
    for w in result.warnings:
        print(f'  {w}')

print()
print(f'{"Q#":<4} {"CEFR":<6} {"Type":<16} {"Stem (preview)"}')
print('-' * 76)
for item in result.store.all_items():
    print(f'{item.question_number:<4} {item.target_cefr:<6} '
          f'{item.question_type:<16} {item.stem[:46]}...')


## Cell 12 -- Save Output

In [ ]:
output = {
    'schema': schema.model_dump(),
    'summary': {
        cefr: {
            'accepted': result.store.count_by_cefr(cefr),
            'target':   getattr(schema.subtopics[0], cefr.lower() + '_count')
                        if schema.subtopics else 0,
        }
        for cefr in _CEFR_LEVELS
    },
    'questions': {
        'easy':   [q.model_dump() for q in result.store.easy],
        'medium': [q.model_dump() for q in result.store.medium],
        'hard':   [q.model_dump() for q in result.store.hard],
    },
    'rejected': result.rejected,
    'warnings': result.warnings,
}

out_path = DATA_DIR / 't2t_generator_output.json'
out_path.write_text(json.dumps(output, indent=2, ensure_ascii=False), encoding='utf-8')

print(f'Saved to: {out_path}')
print(f'  Easy   : {len(result.store.easy)} questions')
print(f'  Medium : {len(result.store.medium)} questions')
print(f'  Hard   : {len(result.store.hard)} questions')
print(f'  Rejected: {len(result.rejected)}')
